In [5]:
import os
import torch

In [3]:
N = torch.tensor([[1., 3.],
                  [2., 8.]])

a = N.sum(1, keepdim=True)
b = N.sum(1)
print(a, a.shape)
print(b, b.shape)
print(N / a)
print(N / b)


tensor([[ 4.],
        [10.]]) torch.Size([2, 1])
tensor([ 4., 10.]) torch.Size([2])
tensor([[0.2500, 0.7500],
        [0.2000, 0.8000]])
tensor([[0.2500, 0.3000],
        [0.5000, 0.8000]])


In [86]:
with open('names.txt', 'r+') as file:
    contents = file.read()

split_lines = contents.splitlines()

zipped = []
for name in split_lines:
    added = ['.'] + list(name) + ['.']
    pairs = zip(added, added[1:])
    for pair in pairs:
        zipped.append(pair)

zipped[:10]

char_sorted = sorted(set(''.join(split_lines)))
ch2i = {}
i2ch = {}
ch2i['.'] = 0
i2ch[0] = '.'
for i, char in enumerate(char_sorted):
    ch2i[char] = i + 1
    i2ch[i + 1] = char

b = {}
for tp in zipped:
    b[tp] = b.get(tp, 0) + 1

N = torch.zeros((27, 27), dtype=torch.int32).float()
for k, v in b.items():
    (s, e) = k
    x = ch2i[s]
    y = ch2i[e]
    N[x, y] = v


In [50]:
P = (N + 1) / (N + 1).sum(1, keepdim=True)
# NLL  negative log likelihood
def calculate_loss():
    total = 0
    for pair in zipped:
        x = ch2i[pair[0]]
        y = ch2i[pair[1]]
        p = P[x, y]
        total += -torch.log(p)
    return total / len(zipped)
calculate_loss()

tensor(2.4544)

In [45]:
def generate_name(gen):
    final_char = ''
    ix = 0
    while True:
        times = N[ix]
        index = torch.multinomial(times, num_samples=1, replacement=True, generator=gen)
        ix = index.item()
        ch = i2ch[ix]
        if ix == 0:
            break
        final_char += ch
    return final_char

In [79]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    print(generate_name(g))

cexze
momasurailezitynn
konimittain
llayn
ka
da
staiyaubrtthrigotai
moliellavo
ke
teda


In [82]:
xs = []
ys = []
for pair in zipped:
    xi = ch2i[pair[0]]
    yi = ch2i[pair[1]]
    xs.append(xi)
    ys.append(yi)

# 根据序号生成对应 tensor 位置的数值，类似于二进制表示
tx = torch.tensor(xs)
ty = torch.tensor(ys)

F = torch.nn.functional
xenc = F.one_hot(tx, num_classes=27).float()
# 初始化随机权重
W = torch.randn((27, 27), generator=g, requires_grad=True)

loss = None
probs = None
for _ in range(1000):
    #做矩阵乘法
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    
    loss = -torch.log(probs[torch.arange(len(ty)), ty]).mean() + 0.01 * (W**2).mean()
    # probs[torch.arange(len(ty)), ty]
    W.grad = None
    loss.backward()
    W.data -= 50 * W.grad

loss

tensor(2.4804, grad_fn=<AddBackward0>)

In [80]:
def generate_name_by_nn(gen):
    final_char = ''
    ix = 0
    while True:
        p = probs[ix]
        # print(p)
        index = torch.multinomial(p, num_samples=1, replacement=True, generator=gen)
        ix = index.item()
        ch = i2ch[ix]
        if ix == 0:
            break
        final_char += ch
    return final_char

In [84]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    print(generate_name_by_nn(g))

cexba
mhalkuraiiaziayhnekljlmjtraierlknan
kh
dr
snaiiaebatrhr
gotas
mdziedlkvu
k
pted
jek


In [85]:
# 验证 W[i].exp() / N[i] 的关系
print(W[3].exp() / N[3])

tensor([0.0232, 0.0240,    inf, 0.0242, 0.2388, 0.0238,    inf, 0.1260, 0.0239,
        0.0235, 0.0888, 0.0236, 0.0232,    inf,    inf, 0.0237, 0.2401, 0.0361,
        0.0233, 0.0600, 0.0248, 0.0248,    inf,    inf, 0.0896, 0.0232, 0.0705],
       grad_fn=<DivBackward0>)
